
# 사전 등록 검증: 미분양 월간 경보로 통일 규칙 2차 vs 5차

- **사전 등록 문서:** [`preregistration_unsold_v2_v5.md`](preregistration_unsold_v2_v5.md) — 커밋 `bf76b82` (2026-10-09 08:02 UTC). 미분양 값을 받기 **전**에 작성.
- **분석 코드:** [`unsold_rules.py`](unsold_rules.py) — 커밋 `8bf37e7` (08:14 UTC). 역시 값을 받기 전에 커밋했고 **이후 수정하지 않음**.
- **자료:** 국토교통부 `DT_MLTM_2082` 시·군·구별 미분양(재고), 2000.12~2026.08. 받은 뒤 사전 등록 문서대로 '계' 행만 제외.

이 노트북은 그 코드를 그대로 돌립니다. 결정은 문서에 적은 규칙으로만 내립니다.


In [ ]:

import os, urllib.request, importlib, warnings; warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/quanter02/chelsea-analytics/claude/great-dirac-3wwgzy/experiments/"
for f in ["unsold_rules.py", "kosis_monitor.py"]:
    if not os.path.exists(f): urllib.request.urlretrieve(RAW + f, f)
import pandas as pd, unsold_rules as R; importlib.reload(R)
U = R.build_units(R.load())
phi = R.phi_from(U); dev, clear = R.thresholds(U)
print(f"평가 단위(시군구) {len({k[0] for k in U})}곳 · φ = {phi:.2f} · 이탈 해 |dev| ≥ {dev:.3f}, 정상 해 < {clear:.3f} (보정 기간 2010~2016 분포)")


## 1. 공식 채점 (채점 기간 2017~2025)

In [ ]:

res, rows = {}, []
for rule, label in [("v2", "2차 (현재 채택)"), ("v5", "5차 (후보)"), ("simple", "단순 규칙 (참고, 결정에 안 씀)")]:
    a, m, cal_t, _ = R.pick(U, phi, rule, dev, clear)
    t, g = R.summary(R.evaluate(U, R.TEST, phi, rule, a, m, dev, clear))
    res[rule] = (a, m, t, g)
    rows.append(dict(규칙=label, α=a, m=round(m, 3), 감지율=t["감지율"], 상반기내=t["상반기내_감지율"], 잘못된경보율=t["잘못된경보율"],
                     구간별_최대_잘못된경보=float(g.잘못된경보율.max()), 통과=R.passes(t, g)))
display(pd.DataFrame(rows).set_index("규칙"))
display(pd.concat({k: v[3] for k, v in res.items()}))
print("사전 등록 결정 규칙 →", R.decide(res["v2"][2], res["v2"][3], res["v5"][2], res["v5"][3]))


## 2. 범용 파이프라인과 일치하는지 (매달 자동 실행이 같은 결과를 내는지)

In [ ]:

import kosis_monitor as K; importlib.reload(K)
for rule in ["v2", "v5"]:
    out = K.run(K.TOPICS["미분양"], rule=rule, verbose=False)
    print(rule, "파이프라인:", out["test"]["감지율"], out["test"]["잘못된경보율"], "| 공식:", res[rule][2]["감지율"], res[rule][2]["잘못된경보율"])


## 3. 보조 정보 (결정에 쓰지 않음, 이미 본 자료): 5차를 기존 주제에

In [ ]:

rows = []
for n in ["혼인", "청년 순이동"]:
    for rule in ["v2", "v5"]:
        out = K.run(K.TOPICS[n], rule=rule, verbose=False); t, g = out["test"], out["groups"]
        rows.append(dict(주제=n, 규칙=rule, 감지율=t["감지율"], 잘못된경보율=t["잘못된경보율"],
                         큰단위_감지율=float(g.감지율.iloc[-1]), 큰단위_잘못된경보=float(g.잘못된경보율.iloc[-1])))
pd.DataFrame(rows).set_index(["주제", "규칙"])



## 4. 결과와 해석 (사전 실행)

**공식 채점 (2017~2025, 이탈 해 113건 · 정상 해 447건)**

| 규칙 | 고른 값 | 감지율 | 잘못된 경보 | 구간별 최대 잘못된 경보 | 통과 |
|---|---|---|---|---|---|
| 2차 | α 0.9, m 0 | 33.6% | 2.9% | 6.2% | 통과 |
| **5차** | α 0.99, m 0 | **36.3%** | 3.1% | 6.2% | 통과 |
| 단순 규칙 (참고) | 누적 추정치 ≥ 76.5% | **99.1%** | 7.8% | 9.6% | 통과 |

**사전 등록 결정: 5차 채택** (둘 다 통과, 5차 감지율이 2차보다 높음).

파이프라인 결과가 공식 채점과 정확히 일치합니다. 일치 확인 과정에서 파이프라인 버그 하나를 찾아 고쳤습니다. 시도 합계 행("서울 계")이 이름 정리 과정에서 필터를 빠져나가는 문제였습니다.

**보조 정보 (5차를 기존 주제에, 이미 본 자료)**

| 주제 | 2차 | 5차 |
|---|---|---|
| 청년 순이동 | 81.0% / 6.3% | 81.6% / 7.2% |
| 혼인 | 82.3% / 3.7% (큰 도시 잘못된 경보 16.3%) | **71.0%** / 3.4% (큰 도시 4.7%) |

### 알게 된 것
1. **사전 등록 절차대로 5차가 채택됐습니다.** 다만 2차와의 차이는 작습니다(+2.7%p).
2. **가장 큰 발견은 단순 규칙입니다.** 재고 자료에서는 통일 규칙(2차·5차) 감지율이 34~36%에 그쳤고, 단순 규칙은 99%를 잡았습니다(잘못된 경보 7.8%).
   - 원인은 미분양 변화가 **덩어리로 움직이기 때문**입니다(신규 분양 수백 호가 한 번에 더해짐). 과분산 φ = 49로 매우 커서, 통계적 하한이 지나치게 넓어집니다.
   - 혼인에서는 반대로 단순 규칙이 27%로 크게 못했습니다.
   - 즉 **"통계 하한"이 도움이 되는지는 잡음 모양(분산 모형)에 달려 있습니다.** 분산 모형이 맞으면 통일 규칙이 이기고, 덩어리 변화가 많으면 단순 규칙이 이깁니다.
3. **5차는 혼인 큰 도시 문제를 풀었지만(16% → 5%) 혼인 감지율을 11%p 잃었습니다.** 어느 규칙도 모든 주제에서 이기지 못했습니다.

### 다음 검증 후보 (같은 자료로 더 고치지 않음)
- **6차 후보:** 분산을 상수 φ가 아니라 지역별 과거 변동폭으로 추정하기. 덩어리 변화가 많은 지역은 넓게, 적은 지역은 좁게 잡습니다.
- 다음 새 자료(예: 출생, 시군구 고용)에서 5차와 6차를 사전 등록으로 비교합니다.
